# 02 — Data Cleaning
### Sales Forecasting & Business Analytics Platform — Phase 3

**Input:** the raw files, plus every finding from Notebook 01's Data Quality Report.
**Output:** one clean, merged dataset saved to `data/processed/cleaned_sales_data.csv`,
ready for EDA (Notebook 03) and feature engineering (Notebook 04).


## Objectives

- Resolve every open item in Notebook 01's Data Quality Report.
- Merge `store.csv` metadata onto `train.csv`.
- Document every cleaning decision and its trade-offs not just apply a fix silently.
- Persist the result so later notebooks don't need to repeat this work.


In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.data_loader import load_train, load_store
from src.preprocessing import clean_and_merge
from src.utils import missing_report
from src.config import CLEANED_DATA_FILE

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

**What this does:** reuses `load_train()`/`load_store()` from Phase 2 (same
loading logic, same validation) and imports the new cleaning functions from
`src/preprocessing.py`, built specifically for this phase.


In [2]:
train = load_train()
store = load_store()

print(f"train: {train.shape}")
print(f"store: {store.shape}")

train: (1017209, 9)
store: (1115, 10)


## Step 1 — Run the Cleaning Pipeline

`clean_and_merge()` in `src/preprocessing.py` orchestrates four steps in order:

1. Parse `Date` from text to a real datetime.
2. Clean `store.csv` (resolve the two genuine missing-value gaps).
3. Left-join the cleaned store metadata onto `train`.
4. Flag the "open but zero sales" edge case found in Notebook 01.

We call it once here, then verify each step's effect below rather than trusting it
blindly.


In [3]:
cleaned = clean_and_merge(train, store)

print("Before:", train.shape)
print("After: ", cleaned.shape)
print()
print("Row count unchanged (left join did not inflate or drop rows)?", len(cleaned) == len(train))

Before: (1017209, 9)
After:  (1017209, 21)

Row count unchanged (left join did not inflate or drop rows)? True


In [4]:
cleaned.dtypes

Store                                        int64
DayOfWeek                                    int64
Date                                datetime64[us]
Sales                                        int64
Customers                                    int64
Open                                         int64
Promo                                        int64
StateHoliday                                   str
SchoolHoliday                                int64
StoreType                                      str
Assortment                                     str
CompetitionDistance                        float64
CompetitionOpenSinceMonth                  float64
CompetitionOpenSinceYear                   float64
Promo2                                       int64
Promo2SinceWeek                            float64
Promo2SinceYear                            float64
PromoInterval                                  str
CompetitionDistance_was_missing               bool
CompetitionOpenSince_was_missin

**Observation:** `Date` is now a proper `datetime` type (was text in
Notebook 01) — this is what lets Notebook 03/04 sort, resample, and extract date
parts without extra conversion.


## Step 2 — Verify the Missing-Value Handling

`store.csv` had two genuine gaps at the *store* level: `CompetitionDistance` (a
handful of stores) and `CompetitionOpenSinceMonth/Year` (354 stores). Let's confirm
they're resolved in the merged dataset and look closely at the resulting numbers,
because they'll look surprisingly large at first glance.


In [5]:
missing_report(cleaned, "cleaned (after pipeline)")

--- Missing values: cleaned (after pipeline) ---
                 missing_count  missing_pct
Promo2SinceWeek         508031        49.94
Promo2SinceYear         508031        49.94
PromoInterval           508031        49.94



,missing_count,missing_pct
Promo2SinceWeek,508031,49.94
Promo2SinceYear,508031,49.94
PromoInterval,508031,49.94


**Important interpretation note:** the *only* remaining missing values are
`Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval`  exactly the columns we
deliberately left untouched, because they're missing precisely when `Promo2 == 0`
(not applicable, not broken). `CompetitionDistance` and `CompetitionOpenSinceMonth/Year`
no longer appear in this report at all. Both gaps are fully resolved.


In [6]:
n_distance_imputed = cleaned["CompetitionDistance_was_missing"].sum()
n_open_since_imputed = cleaned["CompetitionOpenSince_was_missing"].sum()

print(f"Rows with imputed CompetitionDistance:            {n_distance_imputed:,}")
print(f"Rows with imputed CompetitionOpenSinceMonth/Year: {n_open_since_imputed:,}")
print()
print(f"Underlying store count (should be 3):   {cleaned.loc[cleaned['CompetitionDistance_was_missing'], 'Store'].nunique()}")
print(f"Underlying store count (should be 354):  {cleaned.loc[cleaned['CompetitionOpenSince_was_missing'], 'Store'].nunique()}")

Rows with imputed CompetitionDistance:            2,642
Rows with imputed CompetitionOpenSinceMonth/Year: 323,348

Underlying store count (should be 3):   3
Underlying store count (should be 354):  354


**Why the row counts look so much bigger than Notebook 01's store-level
counts:** Notebook 01 found 3 stores missing `CompetitionDistance` — but each of
those 3 stores appears roughly 880 times in `train` (once per day, over ~2.5
years). After merging, that store-level gap shows up as thousands of *day-level*
rows. The `nunique()` check above confirms the underlying store counts still
match Notebook 01 exactly (3 and 354) nothing new went wrong, this is just the
expected effect of merging store-level data onto a daily-grain table.

This is also exactly why we added the `_was_missing` flag columns: any model or
analyst can trace an imputed row straight back to "this store's competition
distance was unknown," rather than treating it as equally trustworthy as a real
measurement.


## Step 3 — The "Open but Zero Sales" Edge Case

Notebook 01 found that `Open == 0` always implies `Sales == 0` (tautological), but
also found a small number of rows where the store was **open** yet sold nothing
a genuinely different, more interesting case. Let's look closer.


In [7]:
suspicious = cleaned[cleaned["Suspicious_Zero_Sales"]]
print(f"Suspicious rows (Open==1, Sales==0): {len(suspicious)}")
print()
print("Distribution across stores (top 10):")
print(suspicious["Store"].value_counts().head(10))
print()
print("Distribution across StateHoliday:")
print(suspicious["StateHoliday"].value_counts())

Suspicious rows (Open==1, Sales==0): 54

Distribution across stores (top 10):
Store
28      3
835     2
102     2
1017    2
1100    2
25      2
623     2
983     2
1039    2
665     2
Name: count, dtype: int64

Distribution across StateHoliday:
StateHoliday
0    54
Name: count, dtype: int64


**Interpretation:** this is a small number of rows relative to the ~850K
open-store rows overall not large enough to meaningfully bias a model either
way. A few stores show up twice, but none dominate. **Every single one of these
54 rows has `StateHoliday == '0'`**  meaning holidays do *not* explain this
pattern, contrary to what might seem like a natural guess. That rules out one
hypothesis cleanly rather than leaving it as unverified speculation; the more
likely explanation is store-specific reporting quirks (e.g. a till/register
outage) rather than a genuine demand or calendar effect.

**Decision:** flag, don't drop. `Suspicious_Zero_Sales` is now a permanent column
in the cleaned dataset, so Notebook 05 can test model performance with and
without these rows included, rather than us making that call unilaterally here
without evidence either way.


## Step 4 — Design Clarification: `Open == 0` Rows

Notebook 01's Data Quality Report listed "exclude `Open == 0` rows from
training/evaluation" as a Notebook 02 action item. Having built the actual
pipeline, we're refining that: **these rows are not removed from the cleaned
dataset saved here.**

**Reasoning:** `Open == 0` rows are truthful, not incorrect dropping them
during *cleaning* would silently prevent Notebook 03 (EDA) from analyzing
closed-store patterns (e.g. holiday-driven closures, which is itself a PRD
requirement under Seasonal Analysis). The exclusion is a *modeling-time*
decision, and belongs in Notebook 05, applied via the already-existing `Open`
column immediately before the train/validation split not baked irreversibly
into the saved dataset here.

This is a refinement of the original plan, not a silent contradiction of it: the
exclusion still happens before any model sees the data, just one phase later
than originally sketched.


In [8]:
open_rows = (cleaned["Open"] == 1).sum()
closed_rows = (cleaned["Open"] == 0).sum()
print(f"Open rows:   {open_rows:,} ({open_rows / len(cleaned):.1%})")
print(f"Closed rows: {closed_rows:,} ({closed_rows / len(cleaned):.1%})")
print()
print("No rows removed here -- this is a preview of what Notebook 05's filter will apply.")

Open rows:   844,392 (83.0%)
Closed rows: 172,817 (17.0%)

No rows removed here -- this is a preview of what Notebook 05's filter will apply.


## Step 5 — Final Structure Check


In [9]:
cleaned.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistance_was_missing,CompetitionOpenSince_was_missing,Suspicious_Zero_Sales
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,False,False,False
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",False,False,False
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",False,False,False
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN,False,False,False
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN,False,False,False


In [10]:
print("Final shape:", cleaned.shape)
print()
print("Duplicate rows after cleaning:", cleaned.duplicated().sum())

Final shape: (1017209, 21)



Duplicate rows after cleaning: 0


**Observation:** still zero duplicates — the merge didn't introduce any, as
expected from a clean left join on a unique key.


## Step 6 — Persist the Cleaned Dataset

Saving to `data/processed/` rather than continuing purely in-memory means Notebook
03 (EDA) can start fresh by reading one file, without re-running this entire
pipeline every time this mirrors how a real production pipeline writes
intermediate artifacts between stages.


In [11]:
CLEANED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(CLEANED_DATA_FILE, index=False)

import os
size_mb = os.path.getsize(CLEANED_DATA_FILE) / (1024 * 1024)
print(f"Saved to: {CLEANED_DATA_FILE}")
print(f"File size: {size_mb:.1f} MB")

Saved to: /home/claude/Sales-Forecasting/data/processed/cleaned_sales_data.csv
File size: 88.9 MB


## Data Cleaning Summary Report

| # | Notebook 01 Finding | Resolution |
|---|---|---|
| 1 | `StateHoliday` dtype | Verified non-issue in Notebook 01 no action needed |
| 2 | `CompetitionDistance` missing (3 stores) | Median-imputed + `CompetitionDistance_was_missing` flag added |
| 3 | `CompetitionOpenSinceMonth/Year` missing (354 stores) | Median-imputed + `CompetitionOpenSince_was_missing` flag added |
| 4 | `Promo2SinceWeek/Year`, `PromoInterval` missing (544 stores) | Left as-is structurally not applicable when `Promo2==0`, not imputed |
| 5 | `Open == 0` → `Sales == 0` | **Refined decision:** not filtered here; will be filtered in Notebook 05 immediately before modeling |
| 6 | Open-but-zero-sales rows | Flagged via new `Suspicious_Zero_Sales` column, not dropped |
| 7 | `test.csv` missing `Open` (a few rows) | Noted only doesn't affect our train-based validation approach |
| 8 | `Date` stored as text | Parsed to real `datetime` dtype |
| 9 | Duplicate rows | Re-confirmed zero after merge |
| 10 | Store-ID consistency | Re-confirmed intact after merge |


## Business Observations

- Roughly a third of stores (354 of 1,115) have unknown competition-open dates
  worth surfacing to stakeholders as a real limitation if "competitive pressure"
  ever becomes a headline metric in the dashboard; it's a well-documented estimate
  for a third of stores, not a precise fact.
- The "open but zero sales" rows are rare enough not to be a data-quality
  emergency, but flagging (rather than silently keeping or dropping) means we can
  make an evidence-based call in Notebook 05/07 instead of guessing now.
- Closed-store days make up a meaningful share of the dataset deferring their
  exclusion to modeling time (rather than deleting them now) keeps that context
  available for the Seasonal/Holiday Analysis the PRD requires in Notebook 03.


## Next Steps

Notebook 03 (`03_eda.ipynb`) will load `data/processed/cleaned_sales_data.csv`
directly and produce the PRD's required 10+ business-driven visualizations:
sales trends, the Store/Product(-proxy)/Promotion/Seasonal analyses, and a
correlation heatmap each with a business question, observation, and
recommended action, per the Master Prompt's EDA rules.
